# val_04 — Flagging sessions where the lickometer misses licks

[dynamic-foraging-processing#96](https://github.com/AllenNeuralDynamics/dynamic-foraging-processing/issues/96):
the mouse licks, the video shows it, the lickometer records nothing. This notebook measures how
often that happens and proposes two QC metrics. `val_03_missed_licks.ipynb` is the long-form
workup — threshold sweeps, single-event panels, video clips.

**Method.** A bottom-view camera tracks `tongue_tip_center` with Lightning Pose. A pose lick is a
frame where the tongue tip comes within `d` pixels of a spout, with a 0.1 s refractory period. Each
pose lick either coincides with a lickometer event within 100 ms (matched) or stands alone
(pose-only).

```
miss rate(d) = n_pose_only / (n_matched + n_pose_only)
             = 1 - P(lickometer fires | tongue reached within d px)
```

**What a pose-only event is.** Three things produce one: a lick the lickometer failed to register,
the tongue approaching a spout without contacting it, and a keypoint error. Event timing does not
separate them, so the miss rate works as a session-level screen and a single event needs the video.
§5 gives the full list of what that costs.

**Coverage.** 53 sessions, 17 subjects, 2024-05-31 to 2025-07-03. The session named in #96
(`behavior_856239_2026-07-24_12-50-23`) has no pose output, so it is absent from this set.

1. Setup
2. Tongue-to-spout distance when the lickometer fires
3. Miss rate across sessions
4. Day-to-day range within an animal
5. Limitations
6. Two QC metrics

## 1. Setup

In [ ]:
%matplotlib inline
from pathlib import Path
import glob
import contextlib
import io

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from aind_dynamic_foraging_behavior_video_analysis.kinematics.tongue_kinematics_utils import (
    mask_keypoint_data,
)
from aind_dynamic_foraging_behavior_video_analysis.kinematics.tongue_lickometer_utils import (
    detect_licks,
    filter_timestamps_refractory,
    calculate_metrics_witheventkeys,
)
from aind_dynamic_foraging_behavior_video_analysis.ephys.tongue_ephys import find_session_dir

from plotstyle import apply_style, PALETTE, style_ax, save_fig
apply_style()

In [ ]:
if Path("/root/capsule").exists():
    ENV     = "codeocean"
    SCRATCH = Path("/root/capsule/scratch")
else:
    ENV     = "local"
    SCRATCH = Path("/Users/mib/Documents/Code/kinematics_analysis/data")

IS_CO    = ENV == "codeocean"
FIG_DIR  = SCRATCH / "figures" / "val_04_lickometer_qc"
SAVE_FIG = False

SESSION_DIR     = SCRATCH / "session_analysis_mlk"
EXAMPLE_SESSION = "behavior_716325_2024-05-31_10-31-14"

# Detection parameters.
CONF          = 0.8    # keypoint confidence floor for masking
T_REFRACTORY  = 0.1    # s; suppresses re-detection from jitter around the threshold
MATCH_WINDOW  = 0.1    # s; pose and lickometer events this close are the same lick
LICK_HALFWIN  = 0.1    # s; half-width for measuring distance around a lickometer lick

# D0 is the operating distance for every session-level number below. 10 px is 2.7x the
# tongue_tip_center error of 3.67 px (pixel_error.ipynb) — tight enough that a pose lick
# means the tongue arrived at the spout. SWEEP is for the calibration curve only.
D0    = 10
SWEEP = [10, 15, 20, 25, 30, 40]

# QC gates and flags, used in S6.
MIN_N_POSE         = 300    # qualifying events needed before a session is ranked
MIN_LICK_COVERAGE  = 0.90   # fraction of lickometer licks with a tracked tongue nearby
FLAG_MISS          = 0.10   # flag a session when the Wilson lower bound clears this
FLAG_SPREAD        = 0.10   # flag a subject when its day-to-day range clears this

C_POSE    = PALETTE["pos"]       # pose-only / flagged
C_LICKO   = PALETTE["neg"]       # lickometer
C_NEUTRAL = PALETTE["not_sig"]   # unflagged

print("ENV={}".format(ENV))

In [ ]:
def load_session(inter):
    """Load one session's tongue keypoints, spout positions and lickometer licks.

    Parameters
    ----------
    inter : pathlib.Path
        The session's ``intermediate_data`` directory.

    Returns
    -------
    dict
        ``tongue`` (masked keypoint table on session time, with ``spout_distance``),
        ``spoutL``/``spoutR`` (mean positions, de-mirrored) and ``licks`` (lickometer
        times, session time).
    """
    keypoint_dfs = {
        key: pd.read_parquet(inter / "kps_raw_{}.parquet".format(key))
        for key in ["tongue_tip_center", "spout_l", "spout_r"]
    }
    kins = pd.read_parquet(inter / "tongue_kins.parquet",
                           columns=["time", "time_in_session"])

    tongue = mask_keypoint_data(keypoint_dfs, "tongue_tip_center", confidence_threshold=CONF)
    tongue["time"] = kins["time_in_session"].values

    # NB the bottom camera mirrors left/right, so spout_r holds the left spout.
    spoutL = np.mean(keypoint_dfs["spout_r"][["x", "y"]], 0)
    spoutR = np.mean(keypoint_dfs["spout_l"][["x", "y"]], 0)

    # Distance to the nearer spout, per frame. Untracked frames stay NaN.
    tracked = tongue[["x", "y"]].notna().all(axis=1)
    xy = tongue.loc[tracked, ["x", "y"]].to_numpy()
    dL = np.linalg.norm(xy - np.array([spoutL["x"], spoutL["y"]]), axis=1)
    dR = np.linalg.norm(xy - np.array([spoutR["x"], spoutR["y"]]), axis=1)
    tongue["spout_distance"] = np.nan
    tongue.loc[tracked, "spout_distance"] = np.minimum(dL, dR)

    return {
        "tongue": tongue, "spoutL": spoutL, "spoutR": spoutR,
        "licks": np.sort(
            pd.read_parquet(inter / "nwb_df_licks.parquet")["timestamps"].to_numpy()),
    }


def min_distance_near(times, dist, event_times, halfwidth):
    """Minimum tracked tongue-spout distance within +/- halfwidth of each event.

    Parameters
    ----------
    times, dist : ndarray
        Tracked frames only, sorted by time, no NaN.
    event_times : ndarray
        Times to measure around, in the same base as ``times``.
    halfwidth : float
        Half-width of the search window, in seconds.

    Returns
    -------
    ndarray
        One value per event; NaN where no tracked frame falls in the window.
    """
    event_times = np.asarray(event_times, dtype=float)
    lo = np.searchsorted(times, event_times - halfwidth, side="left")
    hi = np.searchsorted(times, event_times + halfwidth, side="right")
    out = np.full(event_times.size, np.nan)
    for k in range(event_times.size):
        if hi[k] > lo[k]:
            out[k] = dist[lo[k]:hi[k]].min()
    return out

In [ ]:
def classify_events(sess, d):
    """Detect pose licks at distance ``d`` and score them against the lickometer.

    Parameters
    ----------
    sess : dict
        Output of :func:`load_session`.
    d : float
        Tongue-to-spout distance that counts as a pose lick, in pixels.

    Returns
    -------
    dict
        ``n_matched``, ``n_pose_only``, ``n_pose``, ``miss_rate`` (pose-only over
        n_pose), ``recall`` (of lickometer licks, the fraction pose also found) and
        ``pose_only_times``.
    """
    pose_licks = detect_licks(sess["tongue"], sess["spoutL"], sess["spoutR"], d)
    pose_licks = filter_timestamps_refractory(pose_licks, T_REFRACTORY)
    licko_licks = filter_timestamps_refractory(list(sess["licks"]), T_REFRACTORY)

    # Pose events are passed first, so the second count is unmatched lickometer events
    # and the third is unmatched pose events (see val_02 for the argument order).
    n_matched, n_lickometer_only, n_pose_only, pose_tbl, _ = (
        calculate_metrics_witheventkeys(pose_licks, licko_licks, time_window=MATCH_WINDOW))

    n_pose = n_matched + n_pose_only
    n_licko = n_matched + n_lickometer_only
    return {
        "d": d, "n_matched": n_matched, "n_pose_only": n_pose_only, "n_pose": n_pose,
        "miss_rate": n_pose_only / n_pose if n_pose > 0 else np.nan,
        "recall": n_matched / n_licko if n_licko > 0 else np.nan,
        "pose_only_times": pose_tbl.loc[
            pose_tbl["Status"] == "False Negative", "Time"].to_numpy(),
    }


def wilson_ci(k, n, z=1.96):
    """Wilson score interval for k successes in n trials. NaN pair when n is 0."""
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    den = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / den
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return centre - half, centre + half

## 2. Tongue-to-spout distance when the lickometer fires

A lickometer event is a contact that certainly happened. The distance from tongue tip to the nearer
spout at those moments says how close pose puts the tongue on a real lick, and therefore what `d`
buys. One example session, `behavior_716325_2024-05-31_10-31-14`.

Distances come from confidence-filtered frames, so `spout_distance` exists only above confidence
0.8 and a window with no surviving frame is dropped.

In [ ]:
if not IS_CO:
    print("[skip] Code Ocean only: needs session_analysis_mlk/<session>/intermediate_data/.")
else:
    inter = find_session_dir(EXAMPLE_SESSION, roots=[SESSION_DIR]) / "intermediate_data"
    with contextlib.redirect_stdout(io.StringIO()):
        S = load_session(inter)
    tongue, licks = S["tongue"], S["licks"]
    duration = float(tongue["time"].max() - tongue["time"].min())

    # Distance at each lickometer lick, and at uniformly drawn times for comparison.
    trk = tongue.loc[tongue["spout_distance"].notna(), ["time", "spout_distance"]]
    trk_t, trk_d = trk["time"].to_numpy(), trk["spout_distance"].to_numpy()
    d_at_lick = min_distance_near(trk_t, trk_d, licks, LICK_HALFWIN)
    rng = np.random.default_rng(0)
    d_at_random = min_distance_near(
        trk_t, trk_d,
        rng.uniform(tongue["time"].min(), tongue["time"].max(), len(licks)), LICK_HALFWIN)

    with contextlib.redirect_stdout(io.StringIO()):
        example_sweep = pd.DataFrame([
            {k: v for k, v in classify_events(S, d).items() if k != "pose_only_times"}
            for d in SWEEP])

    print("Session:          {}".format(EXAMPLE_SESSION))
    print("Frames:           {:,} over {:.0f} s, tracked at conf >= {}: {:.0%}".format(
        len(tongue), duration, CONF, tongue["x"].notna().mean()))
    print("Lickometer licks: {:,} ({:.2f}/s)".format(len(licks), len(licks) / duration))
    print("\nDistance at lickometer licks (px): " + " | ".join(
        "{}th {:.1f}".format(q, np.nanpercentile(d_at_lick, q)) for q in [10, 25, 50, 75, 90]))
    print("Lick windows with no tracked frame:   {:,} / {:,} ({:.1%})".format(
        int(np.isnan(d_at_lick).sum()), len(d_at_lick), np.isnan(d_at_lick).mean()))
    print("Random windows with no tracked frame: {:,} / {:,} ({:.1%})".format(
        int(np.isnan(d_at_random).sum()), len(d_at_random), np.isnan(d_at_random).mean()))
    display(example_sweep.round(4))

In [ ]:
if not IS_CO:
    print("[skip] Figure 1 needs the example session.")
else:
    fig, axs = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")

    # -- panel 1: where the tongue is when the lickometer fires --
    # Densities, not counts: the two samples differ in n after dropping empty windows.
    bins = np.arange(0, 121, 3)
    axs[0].hist(d_at_lick[~np.isnan(d_at_lick)], bins=bins, density=True, color=C_LICKO,
                alpha=0.75, label="at lickometer licks (n={:,})".format(
                    int((~np.isnan(d_at_lick)).sum())))
    axs[0].hist(d_at_random[~np.isnan(d_at_random)], bins=bins, density=True, histtype="step",
                lw=1.6, color=PALETTE["neutral"], label="at random tracked times (n={:,})".format(
                    int((~np.isnan(d_at_random)).sum())))
    axs[0].axvline(D0, color=PALETTE["all"], ls="--", lw=1)
    axs[0].annotate("D0 = {} px".format(D0), xy=(D0, 1.0), xycoords=("data", "axes fraction"),
                    xytext=(4, -11), textcoords="offset points", fontsize=8,
                    color=PALETTE["all"])
    axs[0].set_xlabel("Distance to nearer spout (px)")
    axs[0].set_ylabel("Density")
    axs[0].set_title("Tongue position at confirmed licks")
    axs[0].legend(fontsize=7, loc="upper right")

    # -- panel 2: what d costs and what it buys --
    axs[1].plot(example_sweep["d"], example_sweep["miss_rate"], "o-", color=C_POSE,
                label="miss rate: of pose licks,\nfraction with no lickometer event")
    axs[1].plot(example_sweep["d"], example_sweep["recall"], "s-", color=C_LICKO,
                label="recall: of lickometer licks,\nfraction pose finds")
    axs[1].axvline(D0, color=PALETTE["all"], ls="--", lw=1)
    axs[1].set_ylim(0, 1.02)
    axs[1].set_xlabel("Distance threshold, d (px)")
    axs[1].set_ylabel("Fraction")
    axs[1].set_title("Operating point, example session")
    axs[1].legend(fontsize=7, loc="center right")

    for ax in axs:
        style_ax(ax)
    save_fig(fig, "fig1_calibration", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()

### Reading the output

The tongue sits a median 11.0 px from the spout at a confirmed lick, with the 90th percentile at
29.9 px. Against a comparison drawn at random tracked times the distribution is concentrated near
the spout, so distance carries the lick.

1 of 5,846 lick windows has no tracked frame, against 69.6% of random windows. Pose sees the tongue
essentially whenever the lickometer fires, which is what makes a pose-only event informative rather
than a gap in tracking.

Raising `d` trades the two rates against each other. At 10 px recall is 0.45 and the miss rate is
0.9%; at 30 px recall is 0.90 and the miss rate is 3.6%. The extra detections at 30 px include
approaches that never contacted, so the added miss rate is partly pose error. Everything below runs
at `D0` = 10 px, where a pose lick means the tongue arrived — at the cost of scoring fewer than half
the licks.

## 3. Miss rate across sessions

`miss_rate(D0)` per session, with a Wilson 95% interval, over all 53 sessions. Two covariates come
along:

- `lick_coverage` — of the lickometer's licks, the fraction with a tracked tongue within ±100 ms.
  A session where pose did not see the licks cannot report a meaningful pose-only count.
- `n_pose` — qualifying events at `D0`. It spans 145 to 3,928 across the set, and below
  `MIN_N_POSE` the Wilson interval is wider than the spread between sessions.

In [ ]:
def session_miss_rate(sdir, thresholds=None):
    """Miss rate curve for one session. None when the intermediates are incomplete.

    Parameters
    ----------
    sdir : str or pathlib.Path
        A session directory under ``SESSION_DIR``.
    thresholds : list of float, optional
        Distances to evaluate. Defaults to ``SWEEP``.

    Returns
    -------
    pandas.DataFrame or None
        One row per threshold, with counts, miss rate, Wilson bounds and the
        session-level covariates.
    """
    thresholds = SWEEP if thresholds is None else thresholds
    inter = Path(sdir) / "intermediate_data"
    needed = ["kps_raw_tongue_tip_center.parquet", "kps_raw_spout_l.parquet",
              "kps_raw_spout_r.parquet", "tongue_kins.parquet", "nwb_df_licks.parquet"]
    if not all((inter / f).exists() for f in needed):
        return None

    sess = load_session(inter)
    tg = sess["tongue"]
    dur = float(tg["time"].max() - tg["time"].min())
    if dur <= 0 or len(sess["licks"]) == 0:
        return None

    # Pose sensitivity given a lick that certainly happened. Conditioning on the lick makes
    # this independent of how much the animal licked, which tracked_frac is not.
    trk = tg.loc[tg["spout_distance"].notna(), ["time", "spout_distance"]]
    d_lick = min_distance_near(trk["time"].to_numpy(), trk["spout_distance"].to_numpy(),
                               sess["licks"], LICK_HALFWIN)
    lick_coverage = float(np.mean(~np.isnan(d_lick)))

    rows = []
    for d in thresholds:
        r = classify_events(sess, d)
        lo, hi = wilson_ci(r["n_pose_only"], r["n_pose"])
        rows.append({"session": Path(sdir).name, "d": d, "duration_s": dur,
                     "lick_coverage": lick_coverage,
                     "tracked_frac": float(tg["x"].notna().mean()),
                     "n_pose": r["n_pose"], "n_matched": r["n_matched"],
                     "n_pose_only": r["n_pose_only"], "miss_rate": r["miss_rate"],
                     "miss_lo": lo, "miss_hi": hi,
                     "misses_per_min": r["n_pose_only"] / (dur / 60.0)})
    return pd.DataFrame(rows)

In [ ]:
MAX_SESSIONS = None   # set to an int to cut the loop short while iterating

if not IS_CO:
    print("[skip] Needs session_analysis_mlk/*/intermediate_data/.")
else:
    session_dirs = sorted(glob.glob(str(SESSION_DIR / "*")))
    if MAX_SESSIONS is not None:
        session_dirs = session_dirs[:MAX_SESSIONS]

    frames = []
    for k, sdir in enumerate(session_dirs, 1):
        try:
            # detect_licks and filter_timestamps_refractory print once per call; muted.
            with contextlib.redirect_stdout(io.StringIO()):
                curve = session_miss_rate(sdir)
        except Exception as exc:
            print("  [{}/{}] {}: FAILED ({})".format(k, len(session_dirs), Path(sdir).name, exc))
            continue
        if curve is None:
            continue
        frames.append(curve)
        a = curve[curve["d"] == D0].iloc[0]
        print("  [{}/{}] {}  miss {:5.2%} [{:.1%},{:.1%}]  n={:<5d} coverage {:.1%}".format(
            k, len(session_dirs), a["session"], a["miss_rate"], a["miss_lo"], a["miss_hi"],
            int(a["n_pose"]), a["lick_coverage"]))

    curves_df = pd.concat(frames, ignore_index=True)

    qc = curves_df[curves_df["d"] == D0].copy()
    qc["subject"] = qc["session"].str.split("_").str[1]
    qc["rankable"] = (qc["n_pose"] >= MIN_N_POSE) & (qc["lick_coverage"] >= MIN_LICK_COVERAGE)
    qc = qc.sort_values(["rankable", "miss_rate"], ascending=[False, False]).reset_index(drop=True)

    print("\n{} sessions, {} subjects. Rankable {} (n_pose >= {}, coverage >= {:.0%}).".format(
        len(qc), qc["subject"].nunique(), int(qc["rankable"].sum()), MIN_N_POSE,
        MIN_LICK_COVERAGE))
    print("Miss rate at {} px: median {:.2%}, range {:.2%}-{:.2%}.".format(
        D0, qc["miss_rate"].median(), qc["miss_rate"].min(), qc["miss_rate"].max()))

In [ ]:
if not IS_CO or not len(curves_df):
    print("[skip] Needs the sweep above.")
else:
    flagged_sessions = qc[qc["rankable"] & (qc["miss_lo"] > FLAG_MISS)]

    fig, axs = plt.subplots(1, 2, figsize=(12, 5), layout="constrained")

    # -- panel 1: miss_rate(d), one line per session --
    hot = set(flagged_sessions["session"])
    for sess, g in curves_df.groupby("session"):
        g = g.sort_values("d")
        is_hot = sess in hot
        axs[0].plot(g["d"], g["miss_rate"], "-o" if is_hot else "-",
                    color=C_POSE if is_hot else C_NEUTRAL,
                    lw=2 if is_hot else 1, ms=4, alpha=1.0 if is_hot else 0.35,
                    zorder=3 if is_hot else 1,
                    label="{} {}".format(sess.split("_")[1], sess.split("_")[2])
                    if is_hot else None)
    axs[0].axvline(D0, color=PALETTE["all"], ls="--", lw=1)
    axs[0].set_xlabel("Distance threshold, d (px)")
    axs[0].set_ylabel("1 - P(lickometer fires | tongue within d)")
    axs[0].set_title("Miss rate vs d, all sessions")
    axs[0].legend(fontsize=7, loc="upper left", framealpha=0.9)

    # -- panel 2: the review queue --
    y = np.arange(len(qc))
    for lbl, mask, col in [("rankable", qc["rankable"], C_POSE),
                           ("gated out", ~qc["rankable"], C_NEUTRAL)]:
        sub = qc[mask]
        if not len(sub):
            continue
        axs[1].errorbar(sub["miss_rate"], y[mask.to_numpy()],
                        xerr=[sub["miss_rate"] - sub["miss_lo"],
                              sub["miss_hi"] - sub["miss_rate"]],
                        fmt="o", ms=4, lw=1, color=col, ecolor=col, alpha=0.85, label=lbl)
    axs[1].axvline(FLAG_MISS, color=PALETTE["all"], ls="--", lw=1,
                   label="flag at {:.0%}".format(FLAG_MISS))
    axs[1].invert_yaxis()
    axs[1].set_yticks([])
    axs[1].set_xlabel("Miss rate at {} px (95% Wilson CI)".format(D0))
    axs[1].set_ylabel("Session, ranked")
    axs[1].set_title("Review queue")
    axs[1].legend(fontsize=7, loc="lower right")

    for ax in axs:
        style_ax(ax)
    save_fig(fig, "fig2_miss_rate_by_session", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()

    print("Sessions whose Wilson lower bound clears {:.0%}:".format(FLAG_MISS))
    display(flagged_sessions[["session", "miss_rate", "miss_lo", "miss_hi", "n_pose",
                              "lick_coverage", "misses_per_min"]].round(4))
    print("Gated out (n_pose < {} or coverage < {:.0%}):".format(MIN_N_POSE, MIN_LICK_COVERAGE))
    display(qc[~qc["rankable"]][["session", "miss_rate", "n_pose", "lick_coverage"]].round(4))
    print("spearman(miss_rate, tracked_frac) = {:+.3f}".format(
        qc.loc[qc["rankable"], "miss_rate"].corr(
            qc.loc[qc["rankable"], "tracked_frac"], method="spearman")))

### Reading the output

The median session sits at 2.8%, and the distribution has a long right tail: 0.0% to 38.2%. Five
sessions have a Wilson lower bound above 10%, and they come from three animals — `791691` on three
of four days, `752014` on one of four, `761038` on one of four. `763590` scores 20.1% and 33.1% on
its two days and is gated out by `n_pose` (214 and 145), so its intervals run 11 and 15 points
wide.

Intervals that straddle the flag line are not evidence of anything. The gap between the flagged
sessions and the rest of the distribution is what carries the result, not the ranking within the
bulk.

Quoted numbers come from the 53-session run of this sweep at `D0` = 10 px. `lick_coverage` is
measured per session here for the first time, so the gate behind those counts is the `n_pose` half;
on the example session coverage was 100.0%.

`spearman(miss_rate, tracked_frac)` is the confound check: `tracked_frac` counts every
tongue-visible frame, so it rises with non-lick tongue movement, and a session with more tongue out
has more chances to produce a pose-only event without the lickometer failing. It ran +0.277 at
10 px and +0.329 at 30 px, so part of the between-session spread travels with behaviour.

## 4. Day-to-day range within an animal

The same statistic, grouped by subject. Sessions in this set run on consecutive days, and a trained
animal's licking changes slowly across them. Camera geometry and spout placement also hold roughly
fixed within an animal, which takes out the scale problem of §5.3.

In [ ]:
if not IS_CO or not len(curves_df):
    print("[skip] Needs the sweep above.")
else:
    by_subject = (qc.groupby("subject")
                    .agg(n_sessions=("miss_rate", "size"), lo=("miss_rate", "min"),
                         hi=("miss_rate", "max"), median=("miss_rate", "median"))
                    .assign(spread=lambda t: t["hi"] - t["lo"])
                    .sort_values("spread", ascending=False))
    multi = by_subject[by_subject["n_sessions"] >= 2]
    flagged_subjects = multi[multi["spread"] > FLAG_SPREAD]

    fig, ax = plt.subplots(figsize=(7, 5.5), layout="constrained")
    for i, (subject, row) in enumerate(multi.iterrows()):
        is_hot = subject in flagged_subjects.index
        col = C_POSE if is_hot else C_NEUTRAL
        ax.plot([row["lo"], row["hi"]], [i, i], "-", color=col, lw=1.5,
                alpha=1.0 if is_hot else 0.5, zorder=2 if is_hot else 1)
        pts = qc.loc[qc["subject"] == subject, "miss_rate"]
        ax.plot(pts, np.full(len(pts), i), "o", ms=5, color=col,
                alpha=1.0 if is_hot else 0.5, zorder=3 if is_hot else 1)
    ax.axvline(FLAG_SPREAD, color=PALETTE["all"], ls="--", lw=1,
               label="spread flag at {:.0f} points".format(100 * FLAG_SPREAD))
    ax.set_yticks(np.arange(len(multi)))
    ax.set_yticklabels(["{} (n={})".format(s, int(r["n_sessions"]))
                        for s, r in multi.iterrows()], fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel("Miss rate at {} px, one dot per session".format(D0))
    ax.set_title("Day-to-day range within an animal")
    ax.legend(fontsize=7, loc="lower right")
    style_ax(ax)
    save_fig(fig, "fig3_within_subject_spread", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()

    display(multi.round(4))
    print("Subjects with a day-to-day range above {:.0f} points: {}".format(
        100 * FLAG_SPREAD, ", ".join(flagged_subjects.index)))

### Reading the output

`791691` runs 38.2%, 1.9%, 31.1%, 13.0% on 24-27 June 2025 — a 36-point swing across four
consecutive days in one animal. `782394` runs 1.6%, 1.2%, 0.8%, 0.0% across its four days, and
ten of the fourteen multi-session animals hold their range under 8 points.

Four animals exceed 10 points: `791691` (36.3), `752014` (14.4), `763590` (13.0), `761038` (12.9).
Three of them are the animals §3 flagged, and `763590` is the one §3's `n_pose` gate dropped.

## 5. Limitations

1. **A pose-only event is a candidate.** Nothing here confirms a missed lick; the video does that,
   and no events have been scored by eye. The pose false-positive rate is therefore the error term
   on every number above, and it is unmeasured — precision is scored against the lickometer, which
   is the instrument under test. `val_03` §6 extracts the clips for the scoring pass.
2. **`D0` cuts both ways.** A lickometer that fails on glancing contacts fails on licks that sit at
   larger `d`, and 10 px excludes them, which reads the miss rate low. A tongue that passes within
   10 px of a spout without contacting counts as a pose lick, which reads it high. The net
   direction is unknown.
3. **Pixel distance is not comparable across sessions.** Jaw-to-endpoint distance at licking
   movements spans 53-95 px across this set, a factor of 1.8 in apparent scale, so a fixed 10 px is
   a different physical distance per session. Between-session ranking carries camera geometry along
   with lickometer performance; within-subject comparison (§4) mostly does not.
4. **Behaviour leaks into the rate.** `spearman(miss_rate, tracked_frac)` = +0.277. Sessions with
   more tongue-out time score worse, and some of that is extra opportunity for a pose-only event
   rather than a failing lickometer.
5. **One camera, session-mean spouts.** Detection rests on `tongue_tip_center` from the bottom view
   against the mean spout position over the session. A spout that moves mid-session biases the
   distance trace for part of it.
6. **#96's own session is absent.** `behavior_856239_2026-07-24` has no pose output, so the
   reported case is not reproduced. These rates come from 53 sessions, 2024-05-31 to 2025-07-03.

## 6. Two QC metrics

### A. Session miss rate at 10 px, gated and bounded

```
gate   n_pose >= 300  and  lick_coverage >= 0.90
flag   Wilson 95% lower bound of miss_rate(10 px) > 0.10
```

One number per session, no trial alignment, computed from intermediates the pose pipeline already
writes. On this set it flags 5 sessions, out of the 51 that clear the `n_pose` gate, and leaves the median
session at 2.8%. The
gate is what makes the flag readable: `lick_coverage` says pose saw the licks, `n_pose` says the
interval is narrower than the between-session spread. A flagged session goes to a human with the
clips for its tightest pose-only events.

Limitations 1, 3 and 4 apply directly: the rate carries the pose false-positive rate, the pixel
scale, and some of the animal's tongue-out time.

### B. Day-to-day range within a subject

```
require  >= 3 sessions for the subject
flag     max(miss_rate) - min(miss_rate) across sessions > 0.10
```

The discriminator A lacks. A is a level and reads high for a session that is dark, badly framed, or
full of tongue movement; those conditions persist across an animal's consecutive days, and an
intermittent hardware fault does not. Holding the animal fixed also holds the camera geometry
roughly fixed, which removes limitation 3. On this set it flags 4 of 14 multi-session animals, adds
`763590` — which A's `n_pose` gate dropped — and points at which day to pull.

Running both is the recommendation: A ranks sessions for review, B says whether a high rate is the
rig or the animal.

In [ ]:
if not IS_CO or not len(curves_df):
    print("[skip] Needs the analysis above.")
else:
    qc_table = qc[["session", "subject", "duration_s", "n_pose", "n_matched", "n_pose_only",
                   "miss_rate", "miss_lo", "miss_hi", "misses_per_min",
                   "lick_coverage", "tracked_frac"]].copy()
    qc_table["d_px"] = D0
    qc_table["gate_pass"] = qc["rankable"].values
    # A: session-level flag.
    qc_table["flag_session"] = qc_table["gate_pass"] & (qc_table["miss_lo"] > FLAG_MISS)
    # B: subject-level flag, broadcast onto the subject's sessions.
    spread = (qc_table.groupby("subject")["miss_rate"]
              .agg(["min", "max", "size"]).rename(columns={"size": "n_sessions"}))
    spread["subject_spread"] = spread["max"] - spread["min"]
    spread["flag_subject"] = (spread["n_sessions"] >= 3) & (spread["subject_spread"] > FLAG_SPREAD)
    qc_table = qc_table.merge(spread[["n_sessions", "subject_spread", "flag_subject"]],
                              left_on="subject", right_index=True, how="left")
    qc_table["flag_any"] = qc_table["flag_session"] | qc_table["flag_subject"]

    FIG_DIR.mkdir(parents=True, exist_ok=True)
    out = FIG_DIR / "val_04_lickometer_qc.csv"
    qc_table.sort_values("miss_rate", ascending=False).to_csv(out, index=False)

    print("wrote {}  ({} sessions)".format(out, len(qc_table)))
    print("  A, session flag:  {}".format(int(qc_table["flag_session"].sum())))
    print("  B, subject flag:  {} sessions across {} subjects".format(
        int(qc_table["flag_subject"].sum()),
        int(qc_table.loc[qc_table["flag_subject"], "subject"].nunique())))
    print("  either:           {}".format(int(qc_table["flag_any"].sum())))
    display(qc_table[qc_table["flag_any"]]
            .sort_values("miss_rate", ascending=False)
            [["session", "miss_rate", "miss_lo", "n_pose", "subject_spread",
              "flag_session", "flag_subject"]].round(4))